## Stage 4: Knowledge Graph-based Context Organization


### Import Necessary Packages and Define Global Variables


In [1]:
import pickle
import json
import networkx as nx
import nltk
import re
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
from typing import Any
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR: Path = Path("data")
KG_DATA: Path = DATA_DIR / "gmq.pickle"
CHUNK_DATA: Path = DATA_DIR / "expanded_chunks.json"
QUESTION_DATA: Path = DATA_DIR / "hotpot_dev_distractor_v1_sample100.json"
JSON_OUTPPUT: Path = DATA_DIR / "umq.json"
PICKLE_OUTPUT: Path = DATA_DIR / "umq.pickle"

PUNCTUATION_REGEX = re.compile(r"[^\w\s]")
STOPWORDS: set[str] = set(nltk.corpus.stopwords.words("english"))

### Read Knowledge Graph and Data


In [2]:
with open(KG_DATA, "rb") as kg_pickle:
    expanded_kg: nx.Graph = pickle.load(kg_pickle)

with open(CHUNK_DATA, "r", encoding="utf-8") as chunk_json:
    raw_data_chunks: list[dict[str, Any]] = json.load(chunk_json)

with open(QUESTION_DATA, "r", encoding="utf-8") as question_json:
    raw_questions: list[dict[str, Any]] = json.load(question_json)

### Preprocessing


In [3]:
def tokenize(document: str) -> str:
    """
    Tokenize a document

    Args:
        document (str): The document

    Returns:
        str: List of tokens of the document, without stopwords, punctuation, and in their base forms, separated by spaces
    """

    normalized_text: str = PUNCTUATION_REGEX.sub("", document.lower()).strip()
    tokenized_text: list[str] = nltk.tokenize.word_tokenize(normalized_text)
    removed_stopwords: list[str] = [
        text for text in tokenized_text if text not in STOPWORDS
    ]

    ps: nltk.stem.PorterStemmer = nltk.stem.PorterStemmer()
    stemmed_text: list[str] = [ps.stem(text) for text in removed_stopwords]

    return " ".join(stemmed_text)

In [4]:
# Change to access chunks by ID and take only the text
data_chunks: dict[str, dict[str, Any]] = {
    chunk["chunk_id"]: chunk for chunk in raw_data_chunks
}

for chunk in tqdm(data_chunks.values(), desc="Updating chunks"):
    chunk.pop("chunk_id")
    chunk["chunk_text"] = tokenize(chunk["chunk_text"])

# Same with question
questions: dict[str, dict[str, Any]] = {
    question["_id"]: question for question in raw_questions
}

for question in tqdm(questions.values(), desc="Updating questions"):
    question.pop("_id")
    question["graph"] = nx.Graph()
    question["question"] = tokenize(question["question"])

Updating questions: 100%|██████████| 100/100 [00:00<00:00, 3728.44it/s]


### Create Corresponding Subgraph for each Chunk


In [5]:
def doc_cosine_similarity(question: str, source: str) -> float:
    """
    Calculate cosine similarity between the query and the source document.
    Args:
        question (str): The query
        source (str): The source document/chunk

    Returns:
        cosine_similarity (float): The cosine similarity between the provided arguments

    The function uses **TF-IDF** to vectorize the documents
    """
    vectorizer: TfidfVectorizer = TfidfVectorizer()
    tfidf_matrix: Any = vectorizer.fit_transform([question, source])
    return cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0, 0]

In [6]:
for head, tail, data in tqdm(expanded_kg.edges(data=True), desc="Enumerating Edges"):
    chunk_info: dict[str, Any] = data_chunks[data["source_chunk_id"]]
    question_info: dict[str, Any] = questions[chunk_info["question_id"]]
    graph: nx.Graph = question_info["graph"]

    data["weight"] = -doc_cosine_similarity(
        question_info["question"], chunk_info["chunk_text"]
    )
    data["question_id"] = chunk_info["question_id"]

    graph.add_edge(head, tail, **data)

Enumerating Edges: 100%|██████████| 164/164 [00:00<00:00, 326.35it/s]


### Perform MST for All of the Subgraphs


In [7]:
umq: nx.Graph = nx.Graph()
umq_json: dict[str, list[list[dict[str, str]]]] = defaultdict(list)


for question_id, question_info in tqdm(questions.items(), desc="Processing Subgraphs"):
    mst: nx.Graph = nx.minimum_spanning_tree(question_info["graph"])
    if not mst:
        continue

    for subtree in nx.connected_components(mst):
        cur_subtree: list[dict[str, str]] = []
        for head, tail, data in (
            question_info["graph"].subgraph(subtree).edges(data=True)
        ):
            cur_subtree.append(
                {
                    "head": head,
                    "tail": tail,
                    "relation": data["relation"],
                    "source_chunk_id": data["source_chunk_id"],
                }
            )

            umq.add_edge(head, tail, **data)

        umq_json[question_id].append(cur_subtree)

Processing Subgraphs: 100%|██████████| 100/100 [00:00<00:00, 7999.82it/s]


### Save the Results


In [8]:
with open(JSON_OUTPPUT, "w", encoding="utf-8") as umq_json_output:
    json.dump(umq_json, umq_json_output, indent=2)

with open(PICKLE_OUTPUT, "wb") as umq_pickle_output:
    pickle.dump(umq, umq_pickle_output)